# SōmaGraph — MMPose 動画解析エンジン (Colab)

馬の歩様動画から骨格を推定し、歩様メトリクスを算出する。
技術スタックは [幻獣競馬のブログ](https://note.com/gugenkakeiba/n/n45032876f319) と同じ **RTMDet + HRNet(AnimalPose) トップダウン方式**。

> ランタイム → ランタイムのタイプを変更 → **GPU (T4)** を選択してから実行


## 1. 環境セットアップ (5分ほど)


In [ ]:
!nvidia-smi -L


In [ ]:
%pip install -q -U openmim
!mim install -q mmengine "mmcv>=2.0.1" "mmdet>=3.1.0" "mmpose>=1.1.0"

import mmdet, mmpose
print('mmdet', mmdet.__version__, '/ mmpose', mmpose.__version__)


## 2. エンジン取得 & モデルダウンロード


In [ ]:
!git clone -q https://github.com/dawg2004/somagraph.git
%cd somagraph/engine

from somagraph.pipeline import download_models
download_models()   # RTMDet-M (COCO) + HRNet-W32 (AnimalPose)


## 3. 動画をアップロード


In [ ]:
from google.colab import files
up = files.upload()
video_path = list(up.keys())[0]
print('video:', video_path)


## 4. 解析実行

各フレームで 馬検出 → 骨格推定 を行い、`results/` に
`annotated.mp4` / `keypoints.csv` / `dashboard.json` を出力する。


In [ ]:
from somagraph import SomaGraphEngine, EngineConfig

engine = SomaGraphEngine(EngineConfig(device='cuda:0'))
dashboard = engine.analyze_video(video_path, 'results/')
dashboard


## 5. 骨格オーバーレイ動画を確認


In [ ]:
# ブラウザ再生用に H.264 へ変換して埋め込み表示
!ffmpeg -y -loglevel error -i results/annotated.mp4 -vcodec libx264 -pix_fmt yuv420p results/annotated_h264.mp4

from IPython.display import HTML
from base64 import b64encode
mp4 = open('results/annotated_h264.mp4', 'rb').read()
url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
HTML(f'<video width=720 controls src="{url}"></video>')


## 6. 歩様メトリクス (ダッシュボード用JSON)


In [ ]:
import json
print(json.dumps(dashboard, ensure_ascii=False, indent=2))


In [ ]:
# 蹄の上下動を可視化 (左右差の目視確認)
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('results/keypoints.csv')
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for name, style in [('L_F_Paw', '-'), ('R_F_Paw', '--')]:
    s = df[df.keypoint == name]
    axes[0].plot(s.time_s, s.y, style, label=name)
for name, style in [('L_B_Paw', '-'), ('R_B_Paw', '--')]:
    s = df[df.keypoint == name]
    axes[1].plot(s.time_s, s.y, style, label=name)
axes[0].set_title('前肢 蹄の上下動 (y座標)'); axes[0].legend(); axes[0].invert_yaxis()
axes[1].set_title('後肢'); axes[1].legend(); axes[1].invert_yaxis()
axes[1].set_xlabel('time (s)')
plt.tight_layout(); plt.show()


## 7. 結果をダウンロード


In [ ]:
from google.colab import files
files.download('results/annotated_h264.mp4')
files.download('results/keypoints.csv')
files.download('results/dashboard.json')


---

**パイプライン**: フレーム → RTMDet(馬検出, COCO class 17) → クロップ → HRNet-W32(AnimalPose 20kp) → オーバーレイ + メトリクス

旧版 (DeepLabCut SuperAnimal) は `notebooks/SomaGraph_PoseEngine.ipynb` に残してある。
